# `14 — RMQ / RSQ (static tools + sqrt-decomposition)`

* RMQ: Range Sum Query;
* RMQ: Range Min/Max Query

## **Problem statement**
We have an array $a[0..N-1]$ and queries on a half-open interval **[l, r)**:

- **RSQ(l, r)** = $a[l] + a[l+1] + \ldots + a[r-1]$
- **RMQ(l, r)** = $\min(a[l], a[l+1], \ldots, a[r-1])$

Edge case:
- RSQ(l, l) = 0
- RMQ(l, l) = $+\infty$

## **Static vs dynamic**
- **Static**: array does not change → we can preprocess heavily.
- **Dynamic**: updates happen: `update(i, v)` changes $a[i]$ → we need structures that update efficiently.

## **Complexity summary**
| Structure | Preproc | Query | Update |
|---|---:|---:|---:|
| **Prefix sum (RSQ)** | $O(N)$ | **$O(1)$** | $O(N)$ |
| **Sparse table (RMQ)** | $O(N \log N)$ | **$O(1)$** | $O(N \log N)$ |
| **Sqrt-decomp (RSQ)** | $O(N)$ | $O(\sqrt{N})$ | **$O(1)$** |
| **Sqrt-decomp (RMQ)** | $O(N)$ | $O(\sqrt{N})$ | $O(\sqrt{N})$ |

In [1]:
from dataclasses import dataclass
from typing import List, Self


@dataclass
class RSQPrefixSum:
    a: List[int]
    s: List[int]

    @classmethod
    def build(cls, *, a: List[int], verbose: bool = True) -> "RSQPrefixSum":
        s: List[int] = [0]
        if verbose:
            print("-" * 70)
            print("Building prefix sums s[i] = sum(a[0:i])")
            print("-" * 70)
            print(f"a = {a}")
            print(f"s[0] = 0")

        for i, v in enumerate(a):
            s.append(s[-1] + v)
            if verbose:
                print(f"s[{i+1}] = s[{i}] + a[{i}] = {s[i]} + {v} = {s[i+1]}")

        return cls(a=list(a), s=s)

    def rsq(self: Self, *, l: int, r: int, verbose: bool = True) -> int:
        res: int = self.s[r] - self.s[l]
        if verbose:
            print("-" * 70)
            print(f"RSQ({l}, {r}) = s[{r}] - s[{l}] = {self.s[r]} - {self.s[l]} = {res}")
        return res


a = [0, 1, 2, 5, 8, 3, -1, 2]
rsq = RSQPrefixSum.build(a=a, verbose=True)
_ = rsq.rsq(l=1, r=5, verbose=True)
_ = rsq.rsq(l=3, r=7, verbose=True)



----------------------------------------------------------------------
Building prefix sums s[i] = sum(a[0:i])
----------------------------------------------------------------------
a = [0, 1, 2, 5, 8, 3, -1, 2]
s[0] = 0
s[1] = s[0] + a[0] = 0 + 0 = 0
s[2] = s[1] + a[1] = 0 + 1 = 1
s[3] = s[2] + a[2] = 1 + 2 = 3
s[4] = s[3] + a[3] = 3 + 5 = 8
s[5] = s[4] + a[4] = 8 + 8 = 16
s[6] = s[5] + a[5] = 16 + 3 = 19
s[7] = s[6] + a[6] = 19 + -1 = 18
s[8] = s[7] + a[7] = 18 + 2 = 20
----------------------------------------------------------------------
RSQ(1, 5) = s[5] - s[1] = 16 - 0 = 16
----------------------------------------------------------------------
RSQ(3, 7) = s[7] - s[3] = 18 - 3 = 15


**Notes**: RSQ becomes $O(1)$ because it’s just one subtraction, but any update breaks many prefix sums → update is $O(N)$.

In [6]:
from dataclasses import dataclass
from math import floor, log2, ceil
from typing import List, Self


@dataclass
class RMQSparseTable:
    a: List[int]
    st: List[List[int]]

    @classmethod
    def build(cls, *, a: List[int], verbose: bool = True) -> "RMQSparseTable":
        n: int = len(a)
        if n == 0:
            return cls(a=[], st=[])

        K: int = ceil(log2(n)) + 1
        st: List[List[int]] = [[0] * n for _ in range(K)]
        st[0] = list(a)

        if verbose:
            print("-" * 70)
            print("Building Sparse Table: ST[k][i] = min on [i, i+2^k)")
            print("-" * 70)
            print(f"a = {a}")
            print(f"K = {K}")

        for k in range(1, K):
            step: int = 2 ** (k - 1)
            for i in range(n):
                if i + step < n:
                    st[k][i] = min(st[k - 1][i], st[k - 1][i + step])
                else:
                    st[k][i] = st[k - 1][i]

            if verbose:
                print(f"\nLevel k={k} (range length = {2**k}):")
                print(st[k])

        return cls(a=list(a), st=st)

    def rmq(self: Self, *, l: int, r: int, verbose: bool = True) -> int:
        if l == r:
            return 10**18  # +inf

        k: int = floor(log2(r - l))
        left_min: int = self.st[k][l]
        right_min: int = self.st[k][r - (2 ** k)]
        res: int = min(left_min, right_min)

        if verbose:
            print("-" * 70)
            print(f"RMQ({l}, {r}) with k=floor(log2({r-l}))={k}")
            print(f"Use two blocks of length 2^k={2**k}:")
            print(f"  block1 = [{l}, {l + 2**k})  -> ST[{k}][{l}] = {left_min}")
            print(f"  block2 = [{r - 2**k}, {r}) -> ST[{k}][{r - 2**k}] = {right_min}")
            print(f"Answer = min({left_min}, {right_min}) = {res}")
        return res


a = [0, 1, 2, 5, 8, 3, -1, 2]
rmq = RMQSparseTable.build(a=a, verbose=True)
_ = rmq.rmq(l=1, r=5, verbose=True)
_ = rmq.rmq(l=3, r=7, verbose=True)


----------------------------------------------------------------------
Building Sparse Table: ST[k][i] = min on [i, i+2^k)
----------------------------------------------------------------------
a = [0, 1, 2, 5, 8, 3, -1, 2]
K = 4

Level k=1 (range length = 2):
[0, 1, 2, 5, 3, -1, -1, 2]

Level k=2 (range length = 4):
[0, 1, 2, -1, -1, -1, -1, 2]

Level k=3 (range length = 8):
[-1, -1, -1, -1, -1, -1, -1, 2]
----------------------------------------------------------------------
RMQ(1, 5) with k=floor(log2(4))=2
Use two blocks of length 2^k=4:
  block1 = [1, 5)  -> ST[2][1] = 1
  block2 = [1, 5) -> ST[2][1] = 1
Answer = min(1, 1) = 1
----------------------------------------------------------------------
RMQ(3, 7) with k=floor(log2(4))=2
Use two blocks of length 2^k=4:
  block1 = [3, 7)  -> ST[2][3] = -1
  block2 = [3, 7) -> ST[2][3] = -1
Answer = min(-1, -1) = -1


**Notes**: RMQ is O(1) because any interval is covered by two power-of-two blocks. But updates would require rebuilding many blocks → not dynamic-friendly. 

In [7]:
from dataclasses import dataclass
from math import isqrt
from typing import List, Self


@dataclass
class SqrtDecomposition:
    a: List[int]
    block_size: int
    block_sum: List[int]
    block_min: List[int]

    @classmethod
    def build(cls, *, a: List[int], verbose: bool = True) -> "SqrtDecomposition":
        n: int = len(a)
        k: int = max(1, isqrt(n))  # ~sqrt(n)
        num_blocks: int = (n + k - 1) // k

        block_sum: List[int] = [0] * num_blocks
        block_min: List[int] = [10**18] * num_blocks

        for i, v in enumerate(a):
            b = i // k
            block_sum[b] += v
            block_min[b] = min(block_min[b], v)

        if verbose:
            print("-" * 70)
            print("Sqrt-decomposition build")
            print("-" * 70)
            print(f"a = {a}")
            print(f"block_size = {k}")
            print(f"block_sum = {block_sum}")
            print(f"block_min = {block_min}")

        return cls(a=list(a), block_size=k, block_sum=block_sum, block_min=block_min)

    def _rebuild_block(self: Self, *, b: int) -> None:
        start: int = b * self.block_size
        end: int = min(len(self.a), start + self.block_size)
        self.block_sum[b] = 0
        self.block_min[b] = 10**18
        for i in range(start, end):
            self.block_sum[b] += self.a[i]
            self.block_min[b] = min(self.block_min[b], self.a[i])

    def rsq(self: Self, *, l: int, r: int, verbose: bool = True) -> int:
        n: int = len(self.a)
        res: int = 0

        if verbose:
            print("-" * 70)
            print(f"RSQ sqrt-decomp query on [{l}, {r})")
            print("-" * 70)

        # move l to next block boundary
        while l < r and (l % self.block_size) != 0:
            res += self.a[l]
            if verbose:
                print(f"  left edge: add a[{l}]={self.a[l]} -> res={res}")
            l += 1

        # full blocks
        while l + self.block_size <= r:
            b: int = l // self.block_size
            res += self.block_sum[b]
            if verbose:
                print(f"  full block b={b}: add block_sum={self.block_sum[b]} -> res={res}")
            l += self.block_size

        # right edge
        while l < r:
            res += self.a[l]
            if verbose:
                print(f"  right edge: add a[{l}]={self.a[l]} -> res={res}")
            l += 1

        return res

    def rmq(self: Self, *, l: int, r: int, verbose: bool = True) -> int:
        res: int = 10**18

        if verbose:
            print("-" * 70)
            print(f"RMQ sqrt-decomp query on [{l}, {r})")
            print("-" * 70)

        while l < r and (l % self.block_size) != 0:
            res = min(res, self.a[l])
            if verbose:
                print(f"  left edge: min(res, a[{l}]={self.a[l]}) -> res={res}")
            l += 1

        while l + self.block_size <= r:
            b: int = l // self.block_size
            res = min(res, self.block_min[b])
            if verbose:
                print(f"  full block b={b}: min(res, block_min={self.block_min[b]}) -> res={res}")
            l += self.block_size

        while l < r:
            res = min(res, self.a[l])
            if verbose:
                print(f"  right edge: min(res, a[{l}]={self.a[l]}) -> res={res}")
            l += 1

        return res

    def update(self: Self, *, i: int, v: int, verbose: bool = True) -> None:
        b: int = i // self.block_size
        old: int = self.a[i]
        self.a[i] = v

        # RSQ update can be O(1) by delta (lecture note), RMQ needs rebuild block (O(sqrt N))
        delta: int = v - old
        self.block_sum[b] += delta
        self._rebuild_block(b=b)

        if verbose:
            print("-" * 70)
            print(f"update(i={i}, v={v}) in block b={b}")
            print(f"old={old}, delta={delta}")
            print(f"new a = {self.a}")
            print(f"block_sum = {self.block_sum}")
            print(f"block_min = {self.block_min}")


a = [1, 2, 5, 3, 8, 3, 1, 5, 2, 3, 4, 9, 8, 1, 5]
sd = SqrtDecomposition.build(a=a, verbose=True)

_ = sd.rsq(l=1, r=13, verbose=True)
_ = sd.rmq(l=1, r=13, verbose=True)

sd.update(i=1, v=10, verbose=True)
_ = sd.rsq(l=1, r=13, verbose=True)
_ = sd.rmq(l=1, r=13, verbose=True)

----------------------------------------------------------------------
Sqrt-decomposition build
----------------------------------------------------------------------
a = [1, 2, 5, 3, 8, 3, 1, 5, 2, 3, 4, 9, 8, 1, 5]
block_size = 3
block_sum = [8, 14, 8, 16, 14]
block_min = [1, 3, 1, 3, 1]
----------------------------------------------------------------------
RSQ sqrt-decomp query on [1, 13)
----------------------------------------------------------------------
  left edge: add a[1]=2 -> res=2
  left edge: add a[2]=5 -> res=7
  full block b=1: add block_sum=14 -> res=21
  full block b=2: add block_sum=8 -> res=29
  full block b=3: add block_sum=16 -> res=45
  right edge: add a[12]=8 -> res=53
----------------------------------------------------------------------
RMQ sqrt-decomp query on [1, 13)
----------------------------------------------------------------------
  left edge: min(res, a[1]=2) -> res=2
  left edge: min(res, a[2]=5) -> res=2
  full block b=1: min(res, block_min=3) -> re

**Real-world mapping**:

- Prefix sum (RSQ): “total revenue between days l..r” when data is static.
- Sparse table (RMQ): “minimum temperature between days l..r” when data is static.
- Sqrt-decomposition: dashboards where data changes but you can accept $~√N$ query time.